In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import time
import copy

# ===========================================
# 1. DEVICE AND DATA PREPARATION
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')

# Outlier Cleaning
df.loc[df['AT_load_actual_entsoe_transparency'] < 4000, 'AT_load_actual_entsoe_transparency'] = np.nan
df['AT_load_actual_entsoe_transparency'] = df['AT_load_actual_entsoe_transparency'].interpolate(method='time')

features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df[features].dropna()

scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

# Sliding Window for Daily (24-Hour) Multiple Forecasts
def create_daily_sequences(data, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : (i + lookback), :])
        y_target = data[(i + lookback) : (i + lookback + horizon), :]
        y.append(y_target.flatten()) 
    return np.array(X), np.array(y)

lookback = 48  
horizon = 24   
X, y = create_daily_sequences(data_scaled, lookback, horizon)

train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size : train_size + val_size], y[train_size : train_size + val_size]
X_test, y_test = X[train_size + val_size:], y[train_size + val_size:]

X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).float()
X_val_t = torch.from_numpy(X_val).float()
y_val_t = torch.from_numpy(y_val).float()
X_test_t = torch.from_numpy(X_test).float()

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)

print("Data Set Prepared!")

# ===========================================
# 2. MODEL ARCHITECTURE (GRU Only - Fastest and Efficient)
# ===========================================
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.gru(x, h0)
        return self.fc(out[:, -1, :])

# ===========================================
# 3. TRAINING WITH EARLY STOPPING
# ===========================================
def train_gru(model, patience=5, epochs=30):
    print(f"\n--- GRU Model Training Begins ---")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5) 
    
    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)
        
        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f"Erken Durdurma! Model donduruldu (Epoch {epoch+1}).")
                break
                
    print(f"Eğitim Tamamlandı! Süre: {time.time() - start_time:.0f}s | En İyi Val Loss: {best_val_loss:.5f}")
    
    model.load_state_dict(best_model_weights)
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t.to(device)).cpu().numpy()
    return preds

gru_model = GRUModel(input_dim=3, hidden_dim=64, layer_dim=2, output_dim=72)
gru_preds = train_gru(gru_model)

# ===========================================
# 4. METRICS AND NETWORK FLEXIBILITY (F1-SCORE)
# ===========================================
def inverse_transform_daily(data_72d):
    data_reshaped = data_72d.reshape(-1, 3)
    return scaler.inverse_transform(data_reshaped)

y_test_mw = inverse_transform_daily(y_test)
gru_preds_mw = np.clip(inverse_transform_daily(gru_preds), 0, None)

def calculate_metrics(y_true, y_pred, feature_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    wape = (mae / np.mean(y_true)) * 100 if np.mean(y_true) > 0 else 0
    print(f"[{feature_name}] -> RMSE: {rmse:.2f} MW | MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")

print("\n--- GRU MODEL 24-HOUR DAILY FORECAST ERROR REPORT ---")
calculate_metrics(y_test_mw[:, 0], gru_preds_mw[:, 0], "Güneş")
calculate_metrics(y_test_mw[:, 1], gru_preds_mw[:, 1], "Rüzgar")
calculate_metrics(y_test_mw[:, 2], gru_preds_mw[:, 2], "Tüketim")

net_load_actual = y_test_mw[:, 2] - (y_test_mw[:, 0] + y_test_mw[:, 1])
net_load_gru = gru_preds_mw[:, 2] - (gru_preds_mw[:, 0] + gru_preds_mw[:, 1])

CRITICAL_THRESHOLD = 6000 
kriz_gercek = (net_load_actual > CRITICAL_THRESHOLD).astype(int)
kriz_tahmin = (net_load_gru > CRITICAL_THRESHOLD).astype(int)

print("\n==========================================================")
print("NETWORK FLEXIBILITY AND CRISIS DETECTION REPORT (GRU)")
print("==========================================================")
print(f"Kriz Yakalama (Recall): %{recall_score(kriz_gercek, kriz_tahmin)*100:.2f}")
print(f"Hassasiyet (Precision): %{precision_score(kriz_gercek, kriz_tahmin)*100:.2f}")
print(f"F1-Score (Başarı)     : %{f1_score(kriz_gercek, kriz_tahmin)*100:.2f}")
print("==========================================================")

# ===========================================
# 5. VISUALIZATION
# ===========================================
saat_sayisi = 96 
zaman_ekseni = range(saat_sayisi)
fig, axes = plt.subplots(4, 1, figsize=(16, 20))

axes[0].plot(zaman_ekseni, net_load_actual[:saat_sayisi], label='Actual Net Load', color='black', linewidth=2.5)
axes[0].plot(zaman_ekseni, net_load_gru[:saat_sayisi], label='GRU Estimated Net Load', color='rejection', linestyle='--', linewidth=2)
axes[0].axhline(y=CRITICAL_THRESHOLD, color='orange', linestyle='-.', linewidth=2, label=f'Kriz Eşiği ({CRITICAL_THRESHOLD} MW)')
axes[0].fill_between(zaman_ekseni, CRITICAL_THRESHOLD, net_load_actual[:saat_sayisi], where=(net_load_actual[:saat_sayisi] > CRITICAL_THRESHOLD), color='rejection', alpha=0.2, label='Crisis Moments That Realized')
axes[0].set_title('Şebeke Net Yükü ve Esneklik Krizi Tespiti (İlk 4 Gün)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Megawatt (MW)', fontsize=12)
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle=':', alpha=0.7)

axes[1].plot(zaman_ekseni, y_test_mw[:saat_sayisi, 2], label='Actual Consumption', color='black', linewidth=2)
axes[1].plot(zaman_ekseni, gru_preds_mw[:saat_sayisi, 2], label='GRU Estimate', color='blue', linestyle='--')
axes[1].set_title('Elektrik Tüketimi (Şebeke Yükü)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Megawatt (MW)', fontsize=12)
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle=':', alpha=0.7)

axes[2].plot(zaman_ekseni, y_test_mw[:saat_sayisi, 1], label='True Wind', color='black', linewidth=2)
axes[2].plot(zaman_ekseni, gru_preds_mw[:saat_sayisi, 1], label='GRU Estimate', color='green', linestyle='--')
axes[2].set_title('Rüzgar Enerjisi Üretimi', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Megawatt (MW)', fontsize=12)
axes[2].legend(loc='upper right')
axes[2].grid(True, linestyle=':', alpha=0.7)

axes[3].plot(zaman_ekseni, y_test_mw[:saat_sayisi, 0], label='True Sun', color='black', linewidth=2)
axes[3].plot(zaman_ekseni, gru_preds_mw[:saat_sayisi, 0], label='GRU Estimate', color='orange', linestyle='--')
axes[3].set_title('Güneş Enerjisi Üretimi', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Zaman (Saat)', fontsize=12)
axes[3].set_ylabel('Megawatt (MW)', fontsize=12)
axes[3].legend(loc='upper right')
axes[3].grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import copy

# 1. Device Selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Reading Data
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')

# 3. Only 1 Feature (Consumption) Selection and Cleaning
features_only_load = ['AT_load_actual_entsoe_transparency']
data_load = df[features_only_load].dropna()

# 4. Scaling
scaler_load = MinMaxScaler(feature_range=(-1, 1))
data_scaled_load = scaler_load.fit_transform(data_load.values)

# 5. Sliding Window
def create_daily_sequences(data, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : (i + lookback), :])
        y.append(data[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_daily_sequences(data_scaled_load, 48, 24)

# 6. Split Data into Train/Validation/Test
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = torch.from_numpy(X[:train_size]).float(), torch.from_numpy(y[:train_size]).float()
X_val, y_val = torch.from_numpy(X[train_size : train_size + val_size]).float(), torch.from_numpy(y[train_size : train_size + val_size]).float()
X_test, y_test = torch.from_numpy(X[train_size + val_size:]).float(), torch.from_numpy(y[train_size + val_size:]).float()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

# 7. Sadece 1 Girdi Alan GRU Modeli

class GRU_Single(nn.Module):
    def __init__(self):
        super(GRU_Single, self).__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 24) # Çıktı sadece 24 saatlik tüketim

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

model_a = GRU_Single().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model_a.parameters(), lr=0.001)

# 8. Early Stopping Training Cycle
best_val_loss = float('inf')
best_weights = copy.deepcopy(model_a.state_dict())
patience = 5
epochs_no_improve = 0

print("--- ONLY CONSUMPTION (MODEL A) TRAINING STARTED ---")
for epoch in range(25):
    model_a.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model_a(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        
    model_a.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            val_loss += criterion(model_a(X_batch), y_batch).item()
    val_loss /= len(val_loader)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model_a.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f"Erken Durdurma (Epoch {epoch+1})")
            break

model_a.load_state_dict(best_weights)
model_a.eval()
with torch.no_grad():
    preds = model_a(X_test.to(device)).cpu().numpy()

# 9. Metric Calculation (Ablation Study Result)
preds_mw = scaler_load.inverse_transform(preds)
y_test_mw = scaler_load.inverse_transform(y_test.numpy())

mae = mean_absolute_error(y_test_mw, preds_mw)
wape = (mae / np.mean(y_test_mw)) * 100
print(f"\n[SADECE TÜKETİM] -> MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import copy

# 1. Device Selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Data Reading and Preparation (Returning to the Main Format with 3 Variables)
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df[features].dropna()

# 3. Scaling
scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

# 4. Sliding Window
def create_daily_sequences(data, lookback=48, horizon=24):
    X, y = [], []
    for i in range(len(data) - lookback - horizon + 1):
        X.append(data[i : (i + lookback), :])
        y.append(data[(i + lookback) : (i + lookback + horizon), :].flatten()) 
    return np.array(X), np.array(y)

X, y = create_daily_sequences(data_scaled, 48, 24)

# 5. Data Split (Train / Val / Test)
train_size = int(len(X) * 0.70)
val_size = int(len(X) * 0.15)

X_train, y_train = torch.from_numpy(X[:train_size]).float(), torch.from_numpy(y[:train_size]).float()
X_val, y_val = torch.from_numpy(X[train_size : train_size + val_size]).float(), torch.from_numpy(y[train_size : train_size + val_size]).float()
X_test, y_test = torch.from_numpy(X[train_size + val_size:]).float(), torch.from_numpy(y[train_size + val_size:]).float()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

# ===========================================
# 6. ARCHITECTURAL DEFINITIONS
# ===========================================
class LSTM_Model(nn.Module):
    def __init__(self):
        super(LSTM_Model, self).__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 72) # 3 değişken x 24 saat

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class BiLSTM_Model(nn.Module):
    def __init__(self):
        super(BiLSTM_Model, self).__init__()
        # With bidirectional=True, the model reads data both from past to future and from future to past.
        self.lstm = nn.LSTM(input_size=3, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.fc = nn.Linear(128, 72) # Çift yönlü olduğu için gizli katman boyutu 2 katına (128) çıkar

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# ===========================================
# 7. JOINT TRAINING AND TESTING FUNCTION
# ===========================================
def train_and_evaluate(model, model_name, lr=0.001):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience = 5
    epochs_no_improve = 0
    
    print(f"\n--- {model_name} EĞİTİMİ BAŞLADI (LR: {lr}) ---")
    for epoch in range(25):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f"{model_name}: Erken Durdurma Tetiklendi (Epoch {epoch+1})")
                break
                
    # En iyi ağırlıkları yükle ve test et

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu().numpy()
        
    # Inverse scaling to calculate Consumption (Load) metrics only
    preds_mw = scaler.inverse_transform(preds.reshape(-1, 3)).reshape(preds.shape)
    y_test_mw = scaler.inverse_transform(y_test.numpy().reshape(-1, 3)).reshape(y_test.shape)

    # We pull index 2 (Load) values ​​(0: Sun, 1: Wind, 2: Load)
    preds_load = preds_mw[:, 2::3] 
    y_test_load = y_test_mw[:, 2::3]

    mae = mean_absolute_error(y_test_load, preds_load)
    wape = (mae / np.mean(y_test_load)) * 100
    
    print(f"[{model_name}] Başarıyla Tamamlandı -> MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")

# ===========================================
# 8. COACHING AND CREATING A LEADERBOARD
# ===========================================
lstm_model = LSTM_Model()
bilstm_model = BiLSTM_Model()

train_and_evaluate(lstm_model, "LSTM")
train_and_evaluate(bilstm_model, "Bi-LSTM")